In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import tables
import tqdm.notebook as tqdm

In [2]:
model = AutoModelForSeq2SeqLM.from_pretrained("./models/nllb_teacher").cuda().eval()
tokenizer = AutoTokenizer.from_pretrained("./models/nllb_teacher")

In [3]:
with tables.open_file("./data/tokenized_dataset_synth.hdf5", mode="r") as file:
    source_ids = file.root.source_ids.read()
    target_ids = file.root.target_ids.read()
    source_mask = file.root.source_mask.read()
    target_mask = file.root.target_mask.read()
    synth_ids = file.root.synth_ids.read()
    synth_mask = file.root.synth_mask.read()

In [4]:
def prepend_decoder_start_token(
    target_input_ids: torch.Tensor,
    target_input_mask: torch.Tensor,
    *,
    decoder_start_token_id: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size = target_input_ids.size(0)
    device = target_input_ids.device

    start_ids = torch.full(
        size=(batch_size, 1),
        fill_value=decoder_start_token_id,
        dtype=target_input_ids.dtype,
        device=device,
    )

    start_mask = torch.ones(
        size=(batch_size, 1),
        dtype=target_input_mask.dtype,
        device=device,
    )

    teacher_target_input_ids = torch.cat(
        [start_ids, target_input_ids],
        dim=1,
    )

    teacher_target_input_mask = torch.cat(
        [start_mask, target_input_mask],
        dim=1,
    )

    return teacher_target_input_ids, teacher_target_input_mask

In [9]:
batch_size = 4096

filters = tables.Filters(3, "blosc:lz4")

with tables.open_file("./data/tokenized_dataset_prepped.hdf5", "w") as file, torch.no_grad():
    new_source_ids = file.create_carray(file.root, "source_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    new_target_ids = file.create_carray(file.root, "target_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    new_source_mask = file.create_carray(file.root, "source_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)
    new_target_mask = file.create_carray(file.root, "target_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)
    new_synth_ids = file.create_carray(file.root, "synth_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)
    new_synth_mask = file.create_carray(file.root, "synth_mask", tables.BoolAtom(), shape=source_ids.shape, filters=filters)

    logits_size = (*source_ids.shape, 32)
    teacher_top32_logits = file.create_carray(file.root, "teacher_top32_logits", tables.Float32Atom(), shape=logits_size, filters=filters)
    teacher_top32_ids = file.create_carray(file.root, "teacher_top32_ids", tables.Int32Atom(), shape=logits_size, filters=filters)
    teacher_top32_token_ids = file.create_carray(file.root, "teacher_top32_token_ids", tables.Int32Atom(), shape=source_ids.shape, filters=filters)

    for i in tqdm.trange(0, source_ids.shape[0], batch_size):
        # ids_tensor = torch.from_numpy(source_ids[i:i + batch_size]).cuda()
        # mask_tensor = torch.from_numpy(source_mask[i:i + batch_size]).cuda()
        # outputs = model.generate(input_ids=ids_tensor, attention_mask=mask_tensor, max_length=64, forced_bos_token_id=tokenizer.convert_tokens_to_ids("rus_Cyrl"))
        # if outputs.size(1) < 64:
        #     outputs = torch.concat([outputs, torch.full(size=(outputs.size(0), 64 - outputs.size(1)), fill_value=tokenizer.pad_token_id, dtype=outputs.dtype, device="cuda")], dim=1)
        # mask = outputs.ne(tokenizer.pad_token_id)

        target_ids_tensor = torch.from_numpy(target_ids[i:i + batch_size]).cuda()
        target_mask_tensor = torch.from_numpy(target_mask[i:i + batch_size]).cuda()
        source_ids_tensor = torch.from_numpy(source_ids[i:i + batch_size]).cuda()
        source_mask_tensor = torch.from_numpy(source_mask[i:i + batch_size]).cuda()

        target_ids_batch, target_mask_batch = prepend_decoder_start_token(
            target_ids_tensor,
            target_mask_tensor,
            decoder_start_token_id=model.config.decoder_start_token_id,
        )

        outputs = model(
            input_ids=source_ids_tensor,
            attention_mask=source_mask_tensor,
            decoder_input_ids=target_ids_batch,
            decoder_attention_mask=target_mask_batch,
            use_cache=False
        ).logits[:, 1:, :]

        topk_logits, topk_ids = torch.topk(
            outputs,
            k=32,
            dim=-1,
        )
        

        new_source_ids[i:i + batch_size] = source_ids[i:i + batch_size]
        new_target_ids[i:i + batch_size] = target_ids[i:i + batch_size]
        new_source_mask[i:i + batch_size] = source_mask[i:i + batch_size]
        new_target_mask[i:i + batch_size] = target_mask[i:i + batch_size]
        new_synth_ids[i:i + batch_size] = synth_ids[i:i + batch_size]
        new_synth_mask[i:i + batch_size] = synth_mask[i:i + batch_size]
        teacher_top32_logits[i:i + batch_size] = topk_logits.cpu().numpy()
        teacher_top32_ids[i:i + batch_size] = topk_ids.cpu().numpy()
        teacher_top32_token_ids[i:i + batch_size] = topk_ids.cpu().numpy()[..., 0]

        del outputs
        del topk_ids
        del topk_logits

  0%|          | 0/773527 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from extra import upload
upload("./data/tokenized_dataset_synth.hdf5", "data")